# วิเคราะห์เปรียบเทียบ 2 แบรนด์ด้วย ACSI หรือ TAM — คลิกเดียวได้ครบ
(รวมไฟล์ → Factor Analysis/Cronbach α → t-test/ANOVA → Regression ตามสมมติฐาน → ไฟล์ผลลัพธ์ + สรุป .md)

**ใช้งาน:** วางไฟล์ Excel ของ 2 แบรนด์ไว้โฟลเดอร์เดียวกับ notebook (หรือสคริปต์ `run_analysis.py`) แล้วกด Run All
โปรแกรม **ตรวจโมเดลจากหัวคอลัมน์เอง**: มี `EXP1` = ACSI, มี `PEOU1` = TAM และอ่านชื่อแบรนด์จากข้อ B1

| โมเดล | ตัวแปรแฝง | สมมติฐาน | รหัสแบรนด์ `a0` |
|---|---|---|---|
| ACSI | EXP PQ PV IMG SAT COM LOY (7 ตัว 25 ข้อ) | H1–H12 | 1 = CJ, 2 = 7-Eleven |
| TAM | ATT PU PEOU SN FAM CON PR BI USE (9 ตัว 24 ข้อ) | H1–H9 | 1 = Netflix, 2 = Disney+ |

การตั้งชื่อตัวแปร: `D1–D6 → a1–a6`, `B1–Bn → a7…`, ข้อ Likert เรียงตามตัวแปรแฝง `→ b1…`, factor score `→ x1…`
ผลลัพธ์ทั้งหมดอยู่ในโฟลเดอร์ `results_ACSI` หรือ `results_TAM`

Step 0 (เฉพาะ Google Colab) mount drive แล้วย้ายไปโฟลเดอร์ที่วางไฟล์ ถ้ารันบนเครื่องข้ามไป Step 1

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# import os; os.chdir('/content/drive/MyDrive/Colab Notebooks')

Step 1. ติดตั้งไลบรารี

In [ ]:
%pip install pandas numpy scipy statsmodels openpyxl

Step 2. Config: โมเดล ตัวแปร สมมติฐาน ลำดับตัวเลือก (แก้ได้ที่นี่ที่เดียว)

In [ ]:
import os, re, glob, warnings
from collections import OrderedDict
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
warnings.filterwarnings("ignore")

# --- PCA 1 ปัจจัย / KMO / Cronbach alpha เขียนเองด้วย numpy (เทียบเท่า method="principal" และ Stata pcf)
def pca1(sub):
    R = np.corrcoef(sub.values, rowvar=False)
    w, V = np.linalg.eigh(R); i = np.argmax(w)
    ev = w[i]; ld = V[:, i] * np.sqrt(ev)
    if ld.sum() < 0: ld = -ld                      # ให้ loading เป็นบวก
    weights = np.linalg.solve(R, ld)               # ตัวคูณ factor score (regression method)
    return ld, ev, ev / R.shape[0], weights

def kmo_overall(sub):
    R = np.corrcoef(sub.values, rowvar=False); inv = np.linalg.pinv(R)
    P = -inv / np.sqrt(np.outer(np.diag(inv), np.diag(inv)))
    off = ~np.eye(R.shape[0], dtype=bool)
    return (R[off]**2).sum() / ((R[off]**2).sum() + (P[off]**2).sum())

def cronbach(sub):
    k = sub.shape[1]
    return k / (k - 1) * (1 - sub.var(ddof=1).sum() / sub.sum(axis=1).var(ddof=1))

MODEL          = "auto"  # "auto" (ตรวจจากไฟล์) | "ACSI" | "TAM"
DATA_DIR       = "."     # โฟลเดอร์ที่วางไฟล์ข้อมูลแต่ละแบรนด์
MIN_GROUP_SIZE = 10      # กลุ่มย่อยเล็กกว่านี้จะถูกรวม/ตัดออกตอนทำ t/F-test (n ต่อแบรนด์ ~100 จึงใช้ 10)
SCORE_TYPE     = "factor"  # "factor" = factor score สเกล 0-1 จากข้อมูลรวม (ขั้นตอนเดิมของ notebook เดิม/Stata)
                           # "mean"   = ค่าเฉลี่ยข้อ 1-5 (ตามสไลด์ Class 2)

# ส่วนที่ 1 ข้อมูลทั่วไป D1-D6 (ใช้ชุดเดียวกันทั้ง ACSI และ TAM): (ชื่อ, เป็นลำดับ?, ลำดับตัวเลือกจากน้อยไปมาก)
DEMO_D = OrderedDict([
    ("D1", ("เพศ", False, ["ชาย","หญิง","ไม่ระบุ"])),
    ("D2", ("อายุ", True,  ["ต่ำกว่า 25 ปี","25-34 ปี","35-44 ปี","45-54 ปี","55 ปีขึ้นไป"])),
    ("D3", ("สถานภาพการสมรส", False, ["โสด","สมรส","หย่าร้างหรือหม้าย"])),
    ("D4", ("การศึกษาสูงสุด", True, ["ต่ำกว่าปริญญาตรี","ปริญญาตรี","ปริญญาโท","ปริญญาเอก"])),
    ("D5", ("อาชีพ", False, ["นักเรียนหรือนักศึกษา","พนักงานบริษัทเอกชน","ข้าราชการหรือรัฐวิสาหกิจ","ธุรกิจส่วนตัวหรือค้าขาย","อาชีพอิสระ","อื่น ๆ"])),
    ("D6", ("รายได้เฉลี่ยต่อเดือน", True, ["ต่ำกว่า 15,000 บาท","15,000-25,000 บาท","25,001-35,000 บาท","35,001-50,000 บาท","มากกว่า 50,000 บาท"])),
])

MODEL_SPECS = OrderedDict()

# ---------- ACSI (Class 2 สไลด์ ACSI: 7 ตัวแปร 12 เส้นทาง) ----------
MODEL_SPECS["ACSI"] = dict(
    marker="EXP1",
    brands={"CJ": 1, "7-Eleven": 2},                         # รหัสตัวแปร a0
    constructs=OrderedDict([                                  # ลำดับนี้ = ลำดับ b1.. และ x1..
        ("EXP", "ความคาดหวัง (Expectation)"), ("PQ", "คุณภาพที่รับรู้ (Perceived Quality)"),
        ("PV", "ความคุ้มค่า (Perceived Value)"), ("IMG", "ภาพลักษณ์ (Image)"),
        ("SAT", "ความพึงพอใจ (Satisfaction)"), ("COM", "การร้องเรียน (Complaint)"), ("LOY", "ความภักดี (Loyalty)")]),
    hypotheses=[                                              # (รหัส, ตัวต้น, ตัวตาม, เครื่องหมายที่คาด)
        ("H1","EXP","PQ",+1), ("H2","EXP","PV",+1), ("H3","EXP","SAT",+1), ("H4","PQ","PV",+1), ("H5","PQ","SAT",+1),
        ("H6","PV","SAT",+1), ("H7","IMG","PQ",+1), ("H8","IMG","SAT",+1), ("H9","SAT","LOY",+1),
        ("H10","SAT","COM",-1), ("H11","COM","LOY",-1), ("H12","LOY","IMG",+1)],
    behavior=OrderedDict([                                    # ส่วนที่ 2 พฤติกรรมการใช้ B1-B5
        ("B1", ("ระยะเวลาที่ซื้อ/ใช้", True, ["น้อยกว่า 6 เดือน","6 เดือนถึง 1 ปี","มากกว่า 1 ปีถึง 3 ปี","มากกว่า 3 ปี"])),
        ("B2", ("ความถี่ในการซื้อ/ใช้", True, ["เดือนละไม่เกิน 1 ครั้ง","เดือนละ 2-3 ครั้ง","สัปดาห์ละ 1-2 ครั้ง","สัปดาห์ละ 3 ครั้งขึ้นไป"])),
        ("B3", ("วัตถุประสงค์หลัก", False, ["ใช้เอง","ใช้ร่วมกับครอบครัวหรือเพื่อน","ใช้เพื่อการทำงาน","ซื้อเป็นของฝากหรือของขวัญ"])),
        ("B4", ("ค่าใช้จ่ายเฉลี่ยต่อครั้ง", True, ["ต่ำกว่า 100 บาท","100-299 บาท","300-499 บาท","500-999 บาท","1,000 บาทขึ้นไป"])),
        ("B5", ("ช่องทางที่ใช้บ่อยที่สุด", False, ["หน้าร้านหรือสาขา","เว็บไซต์ของแบรนด์","แอปพลิเคชันของแบรนด์","แพลตฟอร์มอื่น เช่น Shopee Lazada"]))]),
)

# ---------- TAM (Class 2 สไลด์ TAM: 9 ตัวแปร 9 เส้นทาง) ----------
MODEL_SPECS["TAM"] = dict(
    marker="PEOU1",
    brands={"Netflix": 1, "Disney+": 2},
    constructs=OrderedDict([
        ("ATT", "ทัศนคติ (Attitude)"), ("PU", "การรับรู้ประโยชน์ (Perceived Usefulness)"),
        ("PEOU", "การรับรู้ความง่าย (Perceived Ease of Use)"), ("SN", "บรรทัดฐานโดยรวม (Subjective Norm)"),
        ("FAM", "ครอบครัว/เพื่อนสนิท (Family)"), ("CON", "คนที่ติดต่อด้วย (Contacts)"),
        ("PR", "ความปลอดภัยที่รับรู้ (Perceived Risk-Security)"), ("BI", "ความตั้งใจใช้ (Behavioral Intention)"),
        ("USE", "การใช้จริง (Actual Use)")]),
    hypotheses=[
        ("H1","PEOU","PU",+1), ("H2","PEOU","ATT",+1), ("H3","PU","ATT",+1), ("H4","ATT","BI",+1), ("H5","FAM","SN",+1),
        ("H6","CON","SN",+1), ("H7","SN","BI",+1), ("H8","PR","BI",+1), ("H9","BI","USE",+1)],
    behavior=OrderedDict([                                    # ส่วนที่ 2 พฤติกรรมการใช้ B1-B4
        ("B1", ("ระยะเวลาที่ใช้", True, ["น้อยกว่า 6 เดือน","6 เดือนถึง 1 ปี","มากกว่า 1 ปีถึง 3 ปี","มากกว่า 3 ปี"])),
        ("B2", ("ความถี่ในการใช้", True, ["น้อยกว่าสัปดาห์ละครั้ง","สัปดาห์ละ 1-2 ครั้ง","สัปดาห์ละ 3-5 ครั้ง","เกือบทุกวัน"])),
        ("B3", ("เวลาที่ใช้ต่อครั้ง", True, ["ไม่เกิน 5 นาที","6-15 นาที","16-30 นาที","มากกว่า 30 นาที"])),
        ("B4", ("วัตถุประสงค์หลักในการใช้", False, ["ใช้ส่วนตัวในชีวิตประจำวัน","ใช้เพื่อการเรียน","ใช้เพื่อการทำงานหรือธุรกิจ","ใช้เพื่อความบันเทิง"]))]),
)

# ---------- ตรวจโมเดลจากไฟล์ที่วางไว้ ----------
def detect_models():
    found = {m: [] for m in MODEL_SPECS}
    for f in sorted(glob.glob(os.path.join(DATA_DIR, "*.xlsx"))):
        if os.path.basename(f).startswith(("~$", "ACSI_", "TAM_", "Table")):
            continue
        try: hdr = [str(c).strip() for c in pd.read_excel(f, nrows=0).columns]
        except Exception: continue
        for m, sp in MODEL_SPECS.items():
            if any(h.startswith(sp["marker"]) for h in hdr): found[m].append(f)
    return found

found = detect_models()
print("ไฟล์ที่พบ:", {m: [os.path.basename(f) for f in fl] for m, fl in found.items()})
if MODEL == "auto":
    cands = [m for m, fl in found.items() if len(fl) >= 2]
    if len(cands) != 1:
        raise SystemExit(f"ตรวจโมเดลไม่ได้ (พบ {cands}) -> ตั้ง MODEL = 'ACSI' หรือ 'TAM' ใน Step 2 แล้วรันใหม่")
    MODEL = cands[0]
SPEC = MODEL_SPECS[MODEL]
CONSTRUCTS, HYPOTHESES, BRAND_CODES = SPEC["constructs"], SPEC["hypotheses"], SPEC["brands"]
DEMO_SPEC = OrderedDict(list(DEMO_D.items()) + list(SPEC["behavior"].items()))
OUT_PREFIX = f"{MODEL}_"
OUT_DIR = f"results_{MODEL}"
os.makedirs(OUT_DIR, exist_ok=True)
def out(name): return os.path.join(OUT_DIR, OUT_PREFIX + name)
print(f"\nโมเดลที่ใช้: {MODEL}  | ตัวแปรแฝง {len(CONSTRUCTS)} ตัว | สมมติฐาน {len(HYPOTHESES)} ข้อ | ผลลัพธ์ -> {OUT_DIR}/")

# โมเดลถดถอยสร้างจากสมมติฐาน: 1 ตัวตาม = 1 โมเดล (ตัวต้นทุกตัวที่ชี้เข้ามา)
MODELS = OrderedDict()
for _dv in OrderedDict.fromkeys(y for (_h, x, y, _s) in HYPOTHESES):   # เรียงตามลำดับที่ปรากฏในสมมติฐาน
    MODELS[f"M{len(MODELS)+1}"] = (_dv, [x for (_h, x, y, _s) in HYPOTHESES if y == _dv])
for m, (dv, iv) in MODELS.items():
    print(f"{m}: {dv} ~ {' + '.join(iv)}")

Step 3. โหลดข้อมูลทุกแบรนด์ + รวมเป็นไฟล์เดียว + ตั้งชื่อตัวแปรใหม่
- รหัสคอลัมน์อ่านจากหัวข้อ (เช่น `PEOU1.`) · ชื่อแบรนด์อ่านจากหัวข้อ B1
- ตัวแปร `a0` = รหัสแบรนด์ (กำหนดใน `brands` ของโมเดล) · บันทึกไฟล์รวม `*_merged_data.xlsx`

In [ ]:
def code_of(col):
    m = re.match(r"^\s*([A-Za-z]+\d+)\s*[\.\s]", str(col))
    return m.group(1).upper() if m else None

files = found[MODEL]
assert len(files) >= 2, f"พบไฟล์ข้อมูล {MODEL} แค่ {len(files)} ไฟล์ใน {os.path.abspath(DATA_DIR)}"

frames, item_text_raw, brand_files = [], {}, {}
for f in files:
    raw = pd.read_excel(f)
    names = {c: code_of(c) for c in raw.columns}
    # ชื่อแบรนด์จากหัวข้อ B1 ("ท่านซื้อหรือใช้ XXX มานานเท่าใด" หรือ "ท่านใช้ XXX มานานเท่าใด")
    b1 = [c for c, k in names.items() if k == "B1"][0]
    mm = re.search(r"ท่าน(?:ซื้อหรือ)?ใช้\s+(.+?)\s+มานาน", str(b1))
    brand = mm.group(1).strip() if mm else os.path.splitext(os.path.basename(f))[0]
    brand_files[brand] = os.path.basename(f)
    keep = {c: k for c, k in names.items() if k is not None}
    for c, k in keep.items():
        item_text_raw.setdefault(k, str(c).split(".", 1)[-1].strip().replace(brand, "{BRAND}"))
    d = raw[list(keep)].rename(columns=keep)
    d.insert(0, "brand", brand)
    frames.append(d)
    print(f"  {brand}: n={len(d)}  (จาก {os.path.basename(f)})")

common = set.intersection(*[set(fr.columns) for fr in frames])          # ใช้เฉพาะรหัสที่ทุกไฟล์มีร่วมกัน
for fr in frames:
    miss = set(fr.columns) - common
    if miss: print("⚠ ตัดคอลัมน์ที่ไม่ตรงกันระหว่างไฟล์:", sorted(miss))
raw_all = pd.concat([fr[[c for c in fr.columns if c in common]] for fr in frames], ignore_index=True)

# รหัส a0: ตามที่กำหนดใน brands; แบรนด์ที่ไม่อยู่ในรายการได้รหัสถัดไป
brand_code = dict(BRAND_CODES)
for b in brand_files:
    if b not in brand_code:
        brand_code[b] = max(brand_code.values()) + 1
brands = sorted(brand_files, key=lambda b: brand_code[b])
raw_all.insert(0, "a0", raw_all["brand"].map(brand_code))
raw_all = raw_all.sort_values("a0", kind="stable").reset_index(drop=True)
print("รหัส a0:", {brand_code[b]: b for b in brands})

item_codes = OrderedDict()
for k in CONSTRUCTS:
    cs = sorted([c for c in raw_all.columns if re.fullmatch(rf"{k}\d+", c)], key=lambda s: int(re.search(r"\d+", s).group()))
    item_codes[k] = cs
demo_codes = [k for k in DEMO_SPEC if k in raw_all.columns]

# ตั้งชื่อใหม่: D1-D6 -> a1-a6, B1-Bn -> a7.., ข้อ Likert -> b1..
rename = OrderedDict()
for i, k in enumerate(demo_codes, 1):
    rename[k] = f"a{i}"
j = 1
for k, cs in item_codes.items():
    for c in cs:
        rename[c] = f"b{j}"; j += 1
df = raw_all[["a0", "brand"] + list(rename)].rename(columns=rename).copy()
a_cols = [rename[k] for k in demo_codes]
b_groups = OrderedDict((k, [rename[c] for c in cs]) for k, cs in item_codes.items())
b_cols = [c for cs in b_groups.values() for c in cs]
print(f"ตัวแปร: a1-a{len(a_cols)} ({len(a_cols)} ตัว), b1-b{len(b_cols)} ({len(b_cols)} ข้อ)")

def to_num(v):
    if pd.isna(v): return np.nan
    if isinstance(v, (int, float, np.number)): return float(v)
    m = re.match(r"^\s*(\d+)", str(v)); return float(m.group(1)) if m else np.nan
for c in b_cols:
    df[c] = df[c].apply(to_num)
bad = df[b_cols].stack()
print("ช่วงค่าข้อ Likert:", bad.min(), "-", bad.max(), "| missing:", int(df[b_cols].isna().sum().sum()))
assert bad.min() >= 1 and bad.max() <= 5, "พบค่านอกช่วง 1-5"

a_label, a_order, a_ordinal, a_code = {}, {}, {}, pd.DataFrame(index=df.index)
for k in demo_codes:
    a = rename[k]; name, ordinal, order = DEMO_SPEC[k]
    df[a] = df[a].astype(str).str.strip()
    extra = [v for v in df[a].unique() if v not in order]
    if extra: print(f"⚠ {k}: พบตัวเลือกนอกรายการในสไลด์ {extra} (ต่อท้ายลำดับ)")
    order = order + extra
    a_label[a] = f"{k}. {name}"; a_order[a] = order; a_ordinal[a] = ordinal
    a_code[a] = df[a].map({v: i + 1 for i, v in enumerate(order)})

a_label["a0"] = "a0. แบรนด์"; a_order["a0"] = brands; a_ordinal["a0"] = False
a_src = {"a0": "brand", **{a: a for a in a_cols}}         # a0 ใช้ข้อความชื่อแบรนด์ในตาราง
a_all = ["a0"] + a_cols
label_ref = pd.DataFrame(
    [{"New": rename[k], "Original": k,
      "Group": "ประชากรศาสตร์/พฤติกรรม" if k in demo_codes else next(g for g, cs in item_codes.items() if k in cs),
      "Text": (DEMO_SPEC[k][0] if k in DEMO_SPEC else item_text_raw.get(k, ""))} for k in rename])
label_ref = pd.concat([pd.DataFrame([{"New": "a0", "Original": "-", "Group": "ตัวแบ่งแบรนด์",
                                      "Text": "แบรนด์: " + ", ".join(f"{brand_code[b]} = {b}" for b in brands)}]), label_ref], ignore_index=True)

# ===== ไฟล์ข้อมูลรวมไฟล์เดียว =====
with pd.ExcelWriter(out("merged_data.xlsx"), engine="openpyxl") as w:
    df[["a0", "brand"] + a_cols + b_cols].to_excel(w, sheet_name="Data_a_b", index=False)          # ชื่อตัวแปรใหม่
    raw_all.to_excel(w, sheet_name="Data_original_codes", index=False)                             # รหัสเดิม D1.., ATT1..
    label_ref.to_excel(w, sheet_name="Codebook", index=False)
print(f"\nรวมข้อมูล: N={len(df)}  a0: {df['a0'].value_counts().sort_index().to_dict()}  -> บันทึก {out('merged_data.xlsx')}")
label_ref.head(40)

Step 4. Factor Analysis (Table 2) บนข้อมูลที่รวมแล้ว — 1 ปัจจัยต่อตัวแปรแฝง (PCA) ได้ loading, %variance, Cronbach's α, AVE, CR
ชีต `Detail` / `Summary` / `Label_Reference` ใช้รูปแบบเดียวกับ Table2 ใน notebook เดิม (ข้อมูลรวม 2 แบรนด์) ส่วนรายแบรนด์ และเกณฑ์ผ่าน/KMO อยู่ในชีตเสริม

In [ ]:
CRIT = dict(alpha=0.70, loading=0.50, ave=0.50, cr=0.70)

def item_desc(it, brand_label):
    return str(label_ref.set_index("New").loc[it, "Text"]).replace("{BRAND}", brand_label)

def run_fa(data, groups):
    detail, summary = [], []
    for k, items in groups.items():
        sub = data[items].dropna()
        ld, ev, prop, _ = pca1(sub)
        alpha = cronbach(sub)
        try: kmo = kmo_overall(sub)
        except Exception: kmo = np.nan
        ave = np.sum(ld**2) / len(ld)
        cr = (ld.sum()**2) / (ld.sum()**2 + np.sum(1 - ld**2))
        ok = (alpha >= CRIT["alpha"]) and (ld.min() >= CRIT["loading"]) and (ave >= CRIT["ave"]) and (cr >= CRIT["cr"])
        for j, it in enumerate(items):
            detail.append({"Construct": k, "Variable": it, "FactorLoading": ld[j]})
        summary.append({"Construct": k, "Name": CONSTRUCTS[k], "N_Items": len(items), "N": len(sub),
                        "Loading_min-max": f"{ld.min():.4f}-{ld.max():.4f}", "Eigenvalue": ev, "%TotalVariance": prop*100,
                        "KMO": kmo, "CronbachAlpha": alpha, "AVE": ave, "CR": cr, "Pass_all(a>=.7,L>=.5,AVE>=.5,CR>=.7)": ok})
    return pd.DataFrame(detail), pd.DataFrame(summary)

def fa_original_format(det, summ, brand_label):
    d_rows, s_rows = [], []
    for k, items in b_groups.items():
        sm = summ[summ.Construct == k].iloc[0]
        for j, it in enumerate(items):
            d_rows.append({"Variable": it, "Label1": CONSTRUCTS[k], "Label2": item_desc(it, brand_label),
                           "FactorLoading": det.loc[det.Variable == it, "FactorLoading"].iloc[0],
                           "%TotalVariance": sm["%TotalVariance"] if j == 0 else np.nan,
                           "CronbachAlpha": sm["CronbachAlpha"] if j == 0 else np.nan,
                           "AVE": sm["AVE"] if j == 0 else np.nan, "CR": sm["CR"] if j == 0 else np.nan})
        s_rows.append({"Factor_No": CONSTRUCTS[k], "N_Items": len(items), "Items": ", ".join(items),
                       "FactorLoading": sm["Loading_min-max"], "%TotalVariance": sm["%TotalVariance"],
                       "CronbachAlpha": sm["CronbachAlpha"], "AVE": sm["AVE"], "CR": sm["CR"]})
    return pd.DataFrame(d_rows).round(4), pd.DataFrame(s_rows).round(4)

def autofit(w, maxw=70):
    for ws in w.book.worksheets:
        for col in ws.columns:
            ws.column_dimensions[col[0].column_letter].width = min(max((len(str(c.value)) if c.value is not None else 0) for c in col) + 2, maxw)

# Pooled = ข้อมูลรวม (ใช้เป็นหลัก) แล้วตามด้วยรายแบรนด์
scopes = OrderedDict([("Pooled", df)] + [(b, df[df["a0"] == brand_code[b]]) for b in brands])
fa_ext, fa_orig = {}, {}
for s, d in scopes.items():
    det, summ = run_fa(d, b_groups)
    fa_ext[s] = summ
    fa_orig[s] = fa_original_format(det, summ, "แบรนด์" if s == "Pooled" else s)
    print(f"\n=== {s} (N={len(d)}) ===")
    print(summ.drop(columns=["Name"]).round(3).to_string(index=False))

with pd.ExcelWriter(out("Table2_FactorAnalysis.xlsx"), engine="openpyxl") as w:
    fa_orig["Pooled"][0].to_excel(w, sheet_name="Detail", index=False)
    fa_orig["Pooled"][1].to_excel(w, sheet_name="Summary", index=False)
    label_ref.to_excel(w, sheet_name="Label_Reference", index=False)
    fa_ext["Pooled"].round(4).to_excel(w, sheet_name="Extended_Pooled", index=False)
    for b in brands:
        fa_orig[b][0].to_excel(w, sheet_name=f"Detail_{b}"[:31], index=False)
        fa_orig[b][1].to_excel(w, sheet_name=f"Summary_{b}"[:31], index=False)
        fa_ext[b].round(4).to_excel(w, sheet_name=f"Extended_{b}"[:31], index=False)
    autofit(w)
print("\n✓ saved", out("Table2_FactorAnalysis.xlsx"))

Step 5. คะแนนตัวแปร (คำนวณจากข้อมูลรวม จึงเทียบข้ามแบรนด์ได้)
- `x1..x7` = factor score สเกล 0–1 (min–max ของข้อมูลรวม) ตามขั้นตอนเดิม · `FA1..FA7` = factor score ดิบ
- `m_XXX` = ค่าเฉลี่ยข้อ (1–5) ตามสไลด์ Class 2 · เลือกชุดที่ใช้วิเคราะห์ด้วย `SCORE_TYPE`

In [ ]:
xn = {k: i for i, k in enumerate(CONSTRUCTS, 1)}          # ตัวแปรแฝงตัวแรก=x1 ...

def factor_scores(data, groups):
    out = pd.DataFrame(index=data.index)
    for k, items in groups.items():
        sub = data[items]; mask = sub.notna().all(axis=1)
        ld, ev, prop, wts = pca1(sub[mask])
        z = (sub[mask] - sub[mask].mean()) / sub[mask].std(ddof=1)
        sc = pd.Series(np.nan, index=data.index); sc[mask] = z.values @ wts
        out[f"FA_{k}"] = sc
        out[f"x_{k}"] = (sc - sc.min()) / (sc.max() - sc.min())
    return out

for k, items in b_groups.items():
    df[f"m_{k}"] = df[items].mean(axis=1)
df = df.join(factor_scores(df, b_groups))
df["brand_d"] = (df["a0"] == max(brand_code.values())).astype(float)   # ตัวควบคุมใน Pooled regression: 1 = แบรนด์รหัสสูงสุด (a0=2), 0 = แบรนด์รหัส 1
scopes = OrderedDict([("Pooled", df)] + [(b, df[df["a0"] == brand_code[b]]) for b in brands])

PREFIX = "m_" if SCORE_TYPE == "mean" else "x_"
S = {k: f"{PREFIX}{k}" for k in CONSTRUCTS}
inv = {v: k for k, v in S.items()}
def xname(col):                                           # ชื่อแสดงผล: x1..x7
    return f"x{xn[inv[col]]}" if col in inv else col
print("SCORE_TYPE =", SCORE_TYPE, "->", {xname(v): v for v in S.values()})
df.groupby("a0")[[S[k] for k in CONSTRUCTS]].mean().round(4)

Step 6. Table 1: Crosstab ของตัวแปร a0–a11 ทุกคู่ (จำนวน แล้วตามด้วยร้อยละแถว) รูปแบบเดียวกับ notebook เดิม
ชีต `Brand_vs_others_chi2` สรุป χ² ของ a0 (แบรนด์) กับตัวแปรอื่น เพื่อดูว่ากลุ่มตัวอย่างสองแบรนด์เทียบกันได้หรือไม่

In [ ]:
import itertools
pairs = list(itertools.combinations(a_all, 2))
print(f"Generating {len(pairs)} crosstab tables ...")
rows = []
for ra, ca in pairs:
    ct = pd.crosstab(df[a_src[ra]], df[a_src[ca]], margins=True, margins_name="รวม")
    ro = [x for x in a_order[ra] if x in ct.index]; co = [x for x in a_order[ca] if x in ct.columns]
    ct = ct.reindex(index=ro + ["รวม"], columns=co + ["รวม"])
    pct = ct.div(ct["รวม"], axis=0) * 100
    rows.append({"Col1": f"Table: {a_label[ra]} vs {a_label[ca]}"})
    hdr = {"Col1": a_label[ra]}
    for i, cn in enumerate(ct.columns): hdr[f"Col{i+2}"] = str(cn)
    rows.append(hdr)
    for idx in ct.index:
        rc, rp = {"Col1": str(idx)}, {"Col1": ""}
        for i, cn in enumerate(ct.columns):
            rc[f"Col{i+2}"] = int(ct.loc[idx, cn]); rp[f"Col{i+2}"] = f"{pct.loc[idx, cn]:.2f}%"
        rows += [rc, rp]
    rows.append({})
crosstab_df = pd.DataFrame(rows)

chi = []
for a in a_cols:
    ct = pd.crosstab(df[a], df["brand"])
    c2, p, dof, exp = stats.chi2_contingency(ct)
    chi.append({"Variable": a_label[a], "chi2": c2, "df": dof, "p": p, "Sig": "*" if p < .05 else "", "%cells_expected<5": (exp < 5).mean() * 100})
chi_df = pd.DataFrame(chi)

with pd.ExcelWriter(out("Table1_Crosstabs.xlsx"), engine="openpyxl") as w:
    crosstab_df.to_excel(w, sheet_name="All_Crosstabs", index=False, header=False)
    chi_df.round(4).to_excel(w, sheet_name="Brand_vs_others_chi2", index=False)
    autofit(w, 30)
print(chi_df.round(3).to_string(index=False))

Step 7. Table 3: ตารางความถี่ ค่าเฉลี่ย S.D. ของข้อ Likert ทั้ง 25 ข้อ (ชีตแรก = ข้อมูลรวม, ตามด้วยรายแบรนด์)

In [ ]:
def freq_table(d, brand_label):
    out = []
    for it in b_cols:
        vc = d[it].value_counts(); n = d[it].count()
        out.append({"ข้อคำถาม": f"{it}: {item_desc(it, brand_label)}", **{str(i): int(vc.get(float(i), 0)) for i in range(1, 6)},
                    "รวม (N)": int(n), "ค่าเฉลี่ย": f"{d[it].mean():.3f}", "S.D.": f"{d[it].std():.3f}"})
        out.append({"ข้อคำถาม": "", **{str(i): f"{vc.get(float(i), 0)/n*100:.1f}%" for i in range(1, 6)}, "รวม (N)": "100.0%", "ค่าเฉลี่ย": "", "S.D.": ""})
    return pd.DataFrame(out)

with pd.ExcelWriter(out("Table3_Frequency_Likert.xlsx"), engine="openpyxl") as w:
    for s, d in scopes.items():
        name = "Frequency_Table" if s == "Pooled" else s[:31]
        freq_table(d, "แบรนด์" if s == "Pooled" else s).to_excel(w, sheet_name=name, index=False)
        w.sheets[name].column_dimensions["A"].width = 80
        for c in "BCDEFGHI": w.sheets[name].column_dimensions[c].width = 12
print("✓ saved Table3 (N pooled =", len(df), ")")

Step 8. Table 4: t-test / F-test (ANOVA)
- **Table4_SmartGrouping / Table4_Standard**: ข้อมูลรวม มีแถว `a0` (แบรนด์) อยู่ด้วย จึงเป็น t-test เทียบ 2 แบรนด์ในทุกตัวแปรแฝง (x1…) (แถวผลกลุ่มอื่น ๆ = ความต่างตามประชากรศาสตร์)
- **Smart grouping**: รวมกลุ่มข้างเคียงที่ n < `MIN_GROUP_SIZE` เฉพาะตัวแปรเรียงลำดับ ส่วนตัวแปรไม่เรียงลำดับ (เพศ อาชีพ ฯลฯ) กลุ่มเล็กไม่นำมาทดสอบ
- **4C**: รายละเอียด (สถิติ df p effect size) · **4D**: Tukey post-hoc เมื่อ ANOVA มีนัยสำคัญ

In [ ]:
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from openpyxl.utils import get_column_letter
def stars(p): return "***" if p < .001 else "**" if p < .01 else "*" if p < .05 else ""

def make_groups(series, a, smart=True):
    cnt = series.value_counts()
    cats = [c for c in a_order[a] if c in cnt.index]
    if not smart:
        return [[c] for c in cats], []
    if a_ordinal[a]:
        grp = [[c] for c in cats]
        while len(grp) > 1:
            ns = [sum(cnt[c] for c in g) for g in grp]; i = int(np.argmin(ns))
            if ns[i] >= MIN_GROUP_SIZE: break
            nb = [j for j in (i-1, i+1) if 0 <= j < len(grp)]
            j = min(nb, key=lambda j: ns[j]); lo, hi = sorted((i, j)); grp[lo] = grp[lo] + grp[hi]; del grp[hi]
        return grp, []
    return [[c] for c in cats if cnt[c] >= MIN_GROUP_SIZE], [c for c in cats if cnt[c] < MIN_GROUP_SIZE]

xcols = [S[k] for k in CONSTRUCTS]
xhdr = {S[k]: f"x{xn[k]}\n({CONSTRUCTS[k].split(' (')[0]})" for k in CONSTRUCTS}

def table4_for(d, smart=True):
    rows = []
    for a in a_all:
        if a == "a0" and d["a0"].nunique() < 2: continue
        grp, dropped = make_groups(d[a_src[a]], a, smart)
        rows.append({"กลุ่มประชากรศาสตร์": f"{a}: {a_label[a].split('. ', 1)[1]}", **{c: "" for c in xcols}})
        data_by_g = []
        for g in grp:
            sub = d[d[a_src[a]].isin(g)]
            lab = (f"รวมกลุ่ม [{', '.join(g)}]" if len(g) > 1 else g[0])
            rows.append({"กลุ่มประชากรศาสตร์": f"   - {lab} (n={len(sub)})", **{c: f"{sub[c].mean():.4f}" for c in xcols}})
            data_by_g.append([sub[c].dropna().values for c in xcols])
        for c_ in dropped:
            rows.append({"กลุ่มประชากรศาสตร์": f"   - {c_} (n={(d[a_src[a]]==c_).sum()}) ไม่นำมาทดสอบ", **{c: "" for c in xcols}})
        r = {"กลุ่มประชากรศาสตร์": f"   สถิติทดสอบ ({'t-test' if len(grp) == 2 else 'F-test'})"}
        for i, c in enumerate(xcols):
            gs = [g[i] for g in data_by_g if len(g[i]) > 1]
            if len(gs) < 2: r[c] = ""; continue
            if len(gs) == 2: st, p = stats.ttest_ind(gs[0], gs[1], equal_var=False); st = abs(st)
            else: st, p = stats.f_oneway(*gs)
            r[c] = f"{st:.3f}{' ' + stars(p) if stars(p) else ''}"
        rows.append(r); rows.append({"กลุ่มประชากรศาสตร์": ""})
    return pd.DataFrame(rows).rename(columns=xhdr)

def test_details(d, scope):
    rows, post = [], []
    for a in a_all:
        if a == "a0" and d["a0"].nunique() < 2: continue
        grp, _ = make_groups(d[a_src[a]], a, True)
        if len(grp) < 2: continue
        labs = [" + ".join(g) for g in grp]
        for k in CONSTRUCTS:
            sm_ = [(l, d.loc[d[a_src[a]].isin(g), S[k]].dropna()) for l, g in zip(labs, grp)]
            sm_ = [(l, v) for l, v in sm_ if len(v) > 1]
            if len(sm_) < 2: continue
            vals = [v for _, v in sm_]
            base = {"Scope": scope, "Variable": a_label[a], "Construct": f"x{xn[k]} {k}",
                    "Groups": " | ".join(f"{l} (n={len(v)}, M={v.mean():.3f})" for l, v in sm_)}
            if len(vals) == 2:
                x, y = vals; vx, vy = x.var(ddof=1)/len(x), y.var(ddof=1)/len(y)
                t, p = stats.ttest_ind(x, y, equal_var=False)
                dfw = (vx + vy)**2 / (vx**2/(len(x)-1) + vy**2/(len(y)-1))
                sp = np.sqrt(((len(x)-1)*x.var(ddof=1) + (len(y)-1)*y.var(ddof=1)) / (len(x)+len(y)-2))
                rows.append({**base, "Test": "Welch t-test", "Statistic": t, "df": f"{dfw:.1f}", "p": p, "Sig": stars(p),
                             "Effect": (x.mean()-y.mean())/sp, "Effect_type": "Cohen d", "Nonparametric_p": stats.mannwhitneyu(x, y).pvalue})
            else:
                F, p = stats.f_oneway(*vals); allv = np.concatenate(vals); gm = allv.mean()
                ssb = sum(len(v)*(v.mean()-gm)**2 for v in vals); sst = ((allv-gm)**2).sum()
                rows.append({**base, "Test": "One-way ANOVA", "Statistic": F, "df": f"{len(vals)-1}, {len(allv)-len(vals)}", "p": p, "Sig": stars(p),
                             "Effect": ssb/sst, "Effect_type": "eta squared", "Nonparametric_p": stats.kruskal(*vals).pvalue})
                if p < .05:
                    tk = pairwise_tukeyhsd(allv, np.concatenate([[l]*len(v) for l, v in sm_]))
                    th = pd.DataFrame(tk.summary().data[1:], columns=tk.summary().data[0])
                    th.insert(0, "Construct", f"x{xn[k]} {k}"); th.insert(0, "Variable", a_label[a]); th.insert(0, "Scope", scope)
                    post.append(th)
    return rows, post

det_rows, post_list = [], []
for sc, dd in scopes.items():
    r_, p_ = test_details(dd, sc); det_rows += r_; post_list += p_
t4c = pd.DataFrame(det_rows)
t4d = pd.concat(post_list, ignore_index=True) if post_list else pd.DataFrame({"note": ["ไม่มี ANOVA ที่มีนัยสำคัญ"]})

def brand_compare(cols):
    rows = []
    for k, nm in CONSTRUCTS.items():
        gs = [df.loc[df["a0"] == brand_code[b], cols[k]].dropna() for b in brands]
        row = {"Variable": f"x{xn[k]}", "Construct": f"{k} {nm}"}
        for b, g in zip(brands, gs): row[f"{b} mean"] = g.mean(); row[f"{b} SD"] = g.std(); row[f"{b} n"] = len(g)
        if len(gs) == 2:
            t, p = stats.ttest_ind(gs[0], gs[1], equal_var=False)
            sp = np.sqrt(((len(gs[0])-1)*gs[0].var() + (len(gs[1])-1)*gs[1].var()) / (len(gs[0])+len(gs[1])-2))
            row.update({"Diff (a0=1 minus a0=2)": gs[0].mean()-gs[1].mean(), "t (Welch)": t, "p": p, "Sig": stars(p),
                        "Cohen_d": (gs[0].mean()-gs[1].mean())/sp, "MannWhitney_p": stats.mannwhitneyu(gs[0], gs[1]).pvalue})
        else:
            F, p = stats.f_oneway(*gs); row.update({"F": F, "p": p, "Sig": stars(p)})
        rows.append(row)
    return pd.DataFrame(rows)
t4a = brand_compare(S)
t4a_mean = brand_compare({k: f"m_{k}" for k in CONSTRUCTS})
print(t4a.round(3).to_string(index=False))
print("\nผลที่มีนัยสำคัญ (p<.05) จากการทดสอบ", len(t4c), "รายการ:")
print(t4c[t4c["p"] < .05][["Scope", "Variable", "Construct", "Test", "Statistic", "df", "p", "Effect"]].round(3).to_string(index=False))

unit = "factor score 0-1" if SCORE_TYPE == "factor" else "ค่าเฉลี่ยข้อ 1-5"
with pd.ExcelWriter(out("Table4_ttest_Ftest_Results.xlsx"), engine="openpyxl") as w:
    for name, d_, smart, title in [("Table4_SmartGrouping", df, True, f"ตารางที่ 4: ความแตกต่างตามประชากรศาสตร์/พฤติกรรม (Smart Grouping รวมกลุ่ม n<{MIN_GROUP_SIZE}, N={len(df)}, {unit})"),
                                   ("Table4_Standard", df, False, f"ตารางที่ 4: ความแตกต่างตามประชากรศาสตร์/พฤติกรรม (ไม่รวมกลุ่ม, N={len(df)}, {unit})")] + \
                                  [(f"Table4_SG_{b}"[:31], scopes[b], True, f"ตารางที่ 4: เฉพาะ {b} (Smart Grouping, N={len(scopes[b])}, {unit})") for b in brands]:
        table4_for(d_, smart).to_excel(w, sheet_name=name, index=False, startrow=1)
        w.sheets[name].cell(row=1, column=1, value=title + "  * p<.05 ** p<.01 *** p<.001")
        w.sheets[name].column_dimensions["A"].width = 55
        for i in range(2, len(xcols) + 2): w.sheets[name].column_dimensions[get_column_letter(i)].width = 20
    t4a.round(4).to_excel(w, sheet_name="4A_BrandCompare", index=False)
    t4a_mean.round(4).to_excel(w, sheet_name="4A_Means_1to5", index=False)
    t4c.round(4).to_excel(w, sheet_name="4C_t_ANOVA_Detail", index=False)
    t4d.round(4).to_excel(w, sheet_name="4D_Tukey_PostHoc", index=False)
    pd.DataFrame([{"Variable": f"x{xn[k]}", "Construct": v, "Column": S[k]} for k, v in CONSTRUCTS.items()]).to_excel(w, sheet_name="Factor_Reference", index=False)
print("✓ saved Table4")

Step 9. Table 5: Regression ตามสมมติฐาน H1…Hn (ตามสมมติฐานของโมเดล) (1 ตัวตาม = 1 โมเดล) รูปแบบเดียวกับ Table5 เดิม (ตัวแปรเป็นแถว โมเดลเป็นคอลัมน์ ค่า b พร้อม * และสถิติ N, R², Adj R², F)
ชีต `Table5` = ข้อมูลรวม (มี `brand_d` ควบคุมแบรนด์) · `Table5_<แบรนด์>` = รายแบรนด์ · Table 6 = สรุปสมมติฐาน

In [ ]:
def fit_scope(d, scope):
    res = OrderedDict()
    for mi, (m, (dv, ivs)) in enumerate(MODELS.items(), 1):
        xs = [S[i] for i in ivs] + (["brand_d"] if scope == "Pooled" else [])
        sub = d[[S[dv]] + xs].dropna()
        X = sm.add_constant(sub[xs]); fit = sm.OLS(sub[S[dv]], X).fit()
        z = (sub - sub.mean()) / sub.std(); zfit = sm.OLS(z[S[dv]], z[xs]).fit()
        vif = max([variance_inflation_factor(X.values, i) for i in range(1, X.shape[1])]) if len(xs) > 1 else 1.0
        res[f"Model{mi}"] = dict(dv=dv, ivs=ivs, xs=xs, fit=fit, beta=zfit.params, n=len(sub), vif=vif, key=m)
    return res

def star3(p): return "***" if p < .01 else "**" if p < .05 else "*" if p < .1 else ""
reg = {s: fit_scope(d, s) for s, d in scopes.items()}
lab1 = {**{S[k]: v for k, v in CONSTRUCTS.items()}, "const": "Constant", "brand_d": f"แบรนด์ (0 = {brands[0]}, 1 = {brands[-1]})"}

def table5_original(scope):
    res = reg[scope]; order = ["const"] + [S[k] for k in CONSTRUCTS] + ["brand_d"]
    used = [v for v in order if any(v in r["fit"].params.index for r in res.values())]
    rows = []
    for v in used:
        row = {"Variable": xname(v), "Label1": lab1[v]}
        for m, r in res.items():
            f = r["fit"]; row[m] = f"{f.params[v]:.4f}{star3(f.pvalues[v])}" if v in f.params.index else ""
        rows.append(row)
    for lab_, fn in [("N", lambda r: str(r["n"])), ("R²", lambda r: f"{r['fit'].rsquared:.4f}"),
                     ("Adj R²", lambda r: f"{r['fit'].rsquared_adj:.4f}"), ("F-statistic", lambda r: f"{r['fit'].fvalue:.4f}")]:
        rows.append({"Variable": lab_, "Label1": lab_, **{m: fn(r) for m, r in res.items()}})
    return pd.DataFrame(rows)

def write_table5(w, scope, sheet):
    t = table5_original(scope); t.to_excel(w, sheet_name=sheet, index=False, startrow=2)
    ws = w.sheets[sheet]; ws.cell(row=1, column=1, value=f"Table5 ({scope}; * p<.1 ** p<.05 *** p<.01; คะแนน={SCORE_TYPE})")
    ws.cell(row=3, column=1, value="Variable"); ws.cell(row=3, column=2, value="Label1")
    for ci, (m, r) in enumerate(reg[scope].items(), 3):
        ws.cell(row=1, column=ci, value=m); ws.cell(row=2, column=ci, value="Dependent_Var"); ws.cell(row=3, column=ci, value=xname(S[r["dv"]]))
    ws.cell(row=2, column=1, value=scope)
    return t

# ---- Table 6: สรุปสมมติฐาน
hy = []
for s, d in scopes.items():
    for h, x, y, sg in HYPOTHESES:
        sub = d[[S[x], S[y]]].dropna(); r_, p_ = stats.pearsonr(sub[S[x]], sub[S[y]])
        rr = next(v for v in reg[s].values() if v["dv"] == y)
        b_, pb_, be_ = rr["fit"].params[S[x]], rr["fit"].pvalues[S[x]], rr["beta"][S[x]]
        res_r = "สนับสนุน" if (np.sign(r_) == sg and p_ < .05) else ("ทิศถูก แต่ไม่มีนัยสำคัญ" if np.sign(r_) == sg else "ทิศตรงข้าม")
        res_b = "สนับสนุน" if (np.sign(b_) == sg and pb_ < .05) else ("ทิศถูก แต่ไม่มีนัยสำคัญ" if np.sign(b_) == sg else "ทิศตรงข้าม")
        hy.append({"Scope": s, "H": h, "Path": f"{x} → {y}", "Vars": f"x{xn[x]} → x{xn[y]}", "Expected": "+" if sg > 0 else "−", "n": len(sub),
                   "r": r_, "p(r)": p_, "Result_r (สไลด์)": res_r, "b": b_, "Std_beta": be_, "p(b)": pb_, "Result_beta (คุมตัวแปรอื่น)": res_b})
t6 = pd.DataFrame(hy)

# ---- ตารางเสริม: b(SE) และ standardized beta ทุก scope
ext = []
for v in ["const"] + [S[k] for k in CONSTRUCTS] + ["brand_d"]:
    r1, r2, rb = {"Variable": xname(v)}, {"Variable": ""}, {"Variable": xname(v)}
    for s in scopes:
        for m, r in reg[s].items():
            c = f"{s}|{m}: {xname(S[r['dv']])}"; f = r["fit"]
            if v in f.params.index:
                r1[c] = f"{f.params[v]:.4f}{star3(f.pvalues[v])}"; r2[c] = f"({f.bse[v]:.4f})"; rb[c] = f"{r['beta'][v]:.3f}" if v in r["beta"].index else ""
    if len(r1) > 1: ext += [r1, r2]
for lab_, fn in [("N", lambda r: str(r["n"])), ("R²", lambda r: f"{r['fit'].rsquared:.4f}"), ("F", lambda r: f"{r['fit'].fvalue:.3f}{star3(r['fit'].f_pvalue)}"), ("max VIF", lambda r: f"{r['vif']:.2f}")]:
    ext.append({"Variable": lab_, **{f"{s}|{m}: {xname(S[r['dv']])}": fn(r) for s in scopes for m, r in reg[s].items()}})

with pd.ExcelWriter(out("Table5_Regression_Results.xlsx"), engine="openpyxl") as w:
    t5 = write_table5(w, "Pooled", "Table5")
    for b in brands: write_table5(w, b, f"Table5_{b}"[:31])
    pd.DataFrame(ext).to_excel(w, sheet_name="Extra_b_SE_VIF", index=False)
    t6.round(4).to_excel(w, sheet_name="Table6_Hypotheses", index=False)
    pd.DataFrame([{"Model": f"Model{i}", "DV": f"x{xn[dv]} {dv}", "IVs": " + ".join(f"x{xn[v]} {v}" for v in iv),
                   "Hypotheses": ", ".join(h for h, x, y, s_ in HYPOTHESES if y == dv)} for i, (m, (dv, iv)) in enumerate(MODELS.items(), 1)]).to_excel(w, sheet_name="Model_Info", index=False)
    pd.DataFrame([{"Variable": f"x{xn[k]}", "Factor_Name": v} for k, v in CONSTRUCTS.items()]).to_excel(w, sheet_name="Factor_Reference", index=False)
    autofit(w, 40)
print(t5.to_string(index=False)); print(); print(t6.round(3).to_string(index=False)); print("✓ saved Table5/6")

Step 10. ส่งออกข้อมูลเป็น Stata (.dta): `a0` (รหัสแบรนด์), a1… เป็นรหัสตามลำดับตัวเลือก, b1…, `FA#` (factor score), `x#` (0-1), `m#` (ค่าเฉลี่ยข้อ)

In [ ]:
st = pd.DataFrame({"a0": df["a0"].astype("int8"), "brand_name": df["brand"]})
for a in a_cols: st[a] = a_code[a].astype("int8")
for c in b_cols: st[c] = df[c]
for k in CONSTRUCTS:
    st[f"FA{xn[k]}"] = df[f"FA_{k}"]; st[f"x{xn[k]}"] = df[f"x_{k}"]; st[f"m{xn[k]}"] = df[f"m_{k}"]
lab = {"a0": "แบรนด์ (" + ", ".join(f"{brand_code[b]}={b}" for b in brands) + ")"}
for a in a_cols: lab[a] = a_label[a][:80]
for _, r in label_ref[label_ref["New"].isin(b_cols)].iterrows(): lab[r["New"]] = f"{r['Original']} {r['Text']}"[:80]
for k, v in CONSTRUCTS.items():
    lab[f"FA{xn[k]}"] = f"factor score {k}"; lab[f"x{xn[k]}"] = f"factor score 0-1 {k}"; lab[f"m{xn[k]}"] = f"mean score {k}"
st.to_stata(out("data2.dta"), write_index=False, version=118, variable_labels={k: v for k, v in lab.items() if k in st.columns})
label_ref.to_excel(out("variable_labels.xlsx"), index=False)
print("✓ saved", out("data2.dta"), st.shape)

Step 11. สรุปผลทั้งหมดเป็นไฟล์ `*_results_preview.md` (Table 1–6 ในรูปแบบเดียวกับ notebook เดิม) เปิดอ่านได้ทันที

In [ ]:
def md_table(d):
    d = d.copy().fillna("")
    cols = [str(c).replace("\n", " ") for c in d.columns]
    s = "| " + " | ".join(cols) + " |\n|" + "|".join(["---"] * len(cols)) + "|\n"
    for _, r in d.iterrows():
        cells = []
        for v in r:
            if isinstance(v, (float, np.floating)): v = (f"{v:.4f}".rstrip("0").rstrip(".")) if abs(v) < 1000 else f"{v:.0f}"
            cells.append(str(v).replace("\n", " ").replace("|", "/"))
        s += "| " + " | ".join(cells) + " |\n"
    return s

L = []; A = L.append
xlist = ", ".join(f"x{xn[k]} {k}" for k in CONSTRUCTS)
A(f"# ผลการวิเคราะห์ {MODEL} (รูปแบบตาราง Table 1–5 ตาม notebook เดิม)\n")
A(f"ข้อมูลรวม {len(brands)} แบรนด์ N = {len(df)} · `a0` " + ", ".join(f"{brand_code[b]} = {b} (n={int((df['a0']==brand_code[b]).sum())})" for b in brands)
  + f" · x1–x{len(CONSTRUCTS)} = {'factor score สเกล 0–1 จากข้อมูลรวม' if SCORE_TYPE=='factor' else 'ค่าเฉลี่ยข้อ 1–5'} ({xlist})\n")
t = pd.read_excel(out("Table1_Crosstabs.xlsx"), sheet_name="All_Crosstabs", header=None)
A("## Table 1: Crosstab (นับ + ร้อยละแถว) — แสดงเฉพาะคู่ที่มี a0 (ครบทุกคู่อยู่ในไฟล์ Excel)\n")
i, n = 0, len(t)
while i < n:
    v = str(t.iloc[i, 0])
    if v.startswith("Table:"):
        j = i
        while j < n and not t.iloc[j].isna().all(): j += 1
        if "a0. แบรนด์ vs" in v:
            blk = t.iloc[i+1:j].copy(); blk.columns = blk.iloc[0]; blk = blk.iloc[1:]; blk = blk.loc[:, blk.columns.notna()]
            A(f"**{v}**\n"); A(md_table(blk))
        i = j
    else: i += 1
A("**χ² ของ a0 (แบรนด์) กับตัวแปรอื่น**\n"); A(md_table(pd.read_excel(out("Table1_Crosstabs.xlsx"), sheet_name="Brand_vs_others_chi2").round(3)))
A("\n## Table 2: Factor analysis (Detail)\n"); A(md_table(pd.read_excel(out("Table2_FactorAnalysis.xlsx"), sheet_name="Detail")))
A("\n**Summary**\n"); A(md_table(pd.read_excel(out("Table2_FactorAnalysis.xlsx"), sheet_name="Summary")))
A("\n## Table 3: ตารางความถี่ (ข้อมูลรวม, 5 = มากที่สุด/เห็นด้วยอย่างยิ่ง)\n")
A(md_table(pd.read_excel(out("Table3_Frequency_Likert.xlsx"), sheet_name="Frequency_Table")))
x4 = pd.read_excel(out("Table4_ttest_Ftest_Results.xlsx"), sheet_name="Table4_SmartGrouping", header=None)
A("\n## Table 4: t-test / F-test (Smart Grouping)\n"); A(str(x4.iloc[0, 0]) + "\n")
b4 = x4.iloc[2:].copy(); b4.columns = x4.iloc[1]; A(md_table(b4))
for sh, tt in [("Table5", "ข้อมูลรวม")] + [(f"Table5_{b}"[:31], b) for b in brands]:
    r5 = pd.read_excel(out("Table5_Regression_Results.xlsx"), sheet_name=sh, header=None)
    A(f"\n## Table 5: Regression — {tt}\n"); A(str(r5.iloc[0, 0]) + "\n")
    hdr = ["Variable", "Label1"] + [f"{r5.iloc[0, c]} (DV: {r5.iloc[2, c]})" for c in range(2, r5.shape[1])]
    body = r5.iloc[3:].copy(); body.columns = hdr; A(md_table(body))
A(f"\n## Table 6: สรุปสมมติฐาน H1–H{len(HYPOTHESES)}\n")
h6 = pd.read_excel(out("Table5_Regression_Results.xlsx"), sheet_name="Table6_Hypotheses")
for sc in ["Pooled"] + brands:
    A(f"**{sc}**\n"); A(md_table(h6[h6.Scope == sc][["H","Vars","Path","Expected","n","r","p(r)","Result_r (สไลด์)","Std_beta","p(b)","Result_beta (คุมตัวแปรอื่น)"]].round(3)))
A(f"\n## เทียบ {brands[0]} กับ {brands[-1]} (Welch t-test)\n")
A(md_table(pd.read_excel(out("Table4_ttest_Ftest_Results.xlsx"), sheet_name="4A_BrandCompare").round(3)))
open(out("results_preview.md"), "w", encoding="utf-8").write("\n".join(L))
print("✓ saved", out("results_preview.md"))
print("\nเสร็จสิ้น — ไฟล์ทั้งหมดอยู่ในโฟลเดอร์:", os.path.abspath(OUT_DIR))
for f in sorted(os.listdir(OUT_DIR)): print("  -", f)